# MIMIC-IV Initial Arrest Rhythm Search

Goal: determine whether the MIMIC-IV cohort contains usable information on initial cardiac arrest rhythm.

This notebook follows the ICU skills pattern: search MIMIC dictionaries first, stream the large event tables by `itemid`, and check coverage in the current analysis cohort. It distinguishes:

- direct charted rhythm-like events near ICU admission, usually `chartevents` telemetry/nursing values;
- admission-level ICD diagnosis evidence for VF/VT/cardiac arrest, which is not a reliable timestamped initial-arrest rhythm;
- existing rhythm-related columns already present in `MIMIC_Predictors.csv`.

Outputs are written under `initial_rhythm_outputs/`, plus a merge-ready `MIMIC_initial_rhythm_features.csv`.

In [1]:
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 80)
warnings.filterwarnings("ignore", category=FutureWarning)

## Configuration

Defaults match the existing MIMIC notebooks in this repository. Override paths on the cluster with environment variables if needed.

In [2]:
# Existing notebooks use /projects/LCICM/mimic-iv-2.2/{hosp,icu}/.
DATABASE_FOLDER = os.environ.get("DATABASE_FOLDER", "/projects/LCICM/")
MIMIC_VERSION = os.environ.get("MIMIC_VERSION", "mimic-iv-2.2")
MIMIC_ROOT = Path(os.environ.get("MIMIC_ROOT", str(Path(DATABASE_FOLDER) / MIMIC_VERSION)))
HOSP = MIMIC_ROOT / "hosp"
ICU = MIMIC_ROOT / "icu"

NOTEBOOK_DIR = Path("mimiciv") if Path("mimiciv").is_dir() and not Path("MIMICAnalysisDML.ipynb").exists() else Path(".")

COHORT_CSV = Path(os.environ.get("MIMIC_COHORT_CSV", str(NOTEBOOK_DIR / "MIMIC_Predictors.csv")))
OUTPUT_DIR = Path(os.environ.get("INITIAL_RHYTHM_OUTPUT_DIR", str(NOTEBOOK_DIR / "initial_rhythm_outputs")))
FEATURES_CSV = NOTEBOOK_DIR / "MIMIC_initial_rhythm_features.csv"
MERGED_PREDICTORS_CSV = NOTEBOOK_DIR / "MIMIC_Predictors_with_initial_rhythm.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = int(os.environ.get("CHUNK_SIZE", "1000000"))
WINDOW_HOURS_PRE = float(os.environ.get("RHYTHM_WINDOW_HOURS_PRE", "6"))
WINDOW_HOURS_POST = float(os.environ.get("RHYTHM_WINDOW_HOURS_POST", "24"))

print("MIMIC_ROOT:", MIMIC_ROOT)
print("COHORT_CSV:", COHORT_CSV.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("FEATURES_CSV:", FEATURES_CSV.resolve())
print(f"Rhythm event window: {-WINDOW_HOURS_PRE:g} to +{WINDOW_HOURS_POST:g} hours from ICU intime")

MIMIC_ROOT: /projects/LCICM/mimic-iv-2.2
COHORT_CSV: /home/mbranda1/ttmhte/mimiciv/MIMIC_Predictors.csv
OUTPUT_DIR: /home/mbranda1/ttmhte/mimiciv/initial_rhythm_outputs
FEATURES_CSV: /home/mbranda1/ttmhte/mimiciv/MIMIC_initial_rhythm_features.csv
Rhythm event window: -6 to +24 hours from ICU intime


In [3]:
def resolve_table(folder, stem):
    candidates = [
        Path(folder) / f"{stem}.csv",
        Path(folder) / f"{stem}.csv.gz",
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"Could not find {stem}.csv or {stem}.csv.gz in {folder}")


def read_table(folder, stem, **kwargs):
    path = resolve_table(folder, stem)
    return pd.read_csv(path, compression="infer", **kwargs)


def table_columns(path):
    return pd.read_csv(path, compression="infer", nrows=0).columns.tolist()


def normalize_text(value):
    if pd.isna(value):
        return ""
    value = str(value).lower()
    value = value.replace("_", " ")
    value = re.sub(r"[^a-z0-9+./ -]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def optional_text_col(df, col):
    if col in df.columns:
        return df[col].astype(str)
    return pd.Series([""] * len(df), index=df.index)

## Load Current MIMIC Analysis Cohort

The current predictors file is used only to define the analysis cohort and to inspect whether rhythm columns are already present.

In [4]:
predictors = pd.read_csv(COHORT_CSV) if COHORT_CSV.exists() else pd.DataFrame()
if predictors.empty:
    print("No predictors file found locally. The notebook can still inspect dictionaries, but cohort-level extraction needs MIMIC_Predictors.csv.")
else:
    print("Predictors shape:", predictors.shape)
    display(predictors.head())

id_cols = [c for c in ["subject_id", "hadm_id", "stay_id"] if c in predictors.columns]
print("Identifier columns in predictors:", id_cols)

rhythm_predictor_cols = [
    c for c in predictors.columns
    if re.search(r"rhythm|vfib|v.fib|vtach|v.tach|ventricular_fibrillation|ventricular_tachycardia|asystole|pulseless|\bpea\b", c, re.I)
]
print(f"Rhythm-like columns already in predictors: {len(rhythm_predictor_cols)}")
display(pd.DataFrame({"column": rhythm_predictor_cols[:200]}))

if rhythm_predictor_cols:
    coverage = []
    for col in rhythm_predictor_cols:
        s = predictors[col]
        nonmissing = s.notna().mean()
        if pd.api.types.is_numeric_dtype(s):
            positive = (pd.to_numeric(s, errors="coerce").fillna(0) != 0).mean()
        else:
            positive = s.astype(str).str.strip().ne("").mean()
        coverage.append({"column": col, "nonmissing": nonmissing, "positive_or_nonempty": positive})
    predictor_rhythm_coverage = pd.DataFrame(coverage).sort_values("positive_or_nonempty", ascending=False)
    display(predictor_rhythm_coverage.head(80))
    predictor_rhythm_coverage.to_csv(OUTPUT_DIR / "existing_predictor_rhythm_column_coverage.csv", index=False)

Predictors shape: (743, 7886)


,subject_id,gender,age,death_at_disch,chart_first_absolute_count_-_basos,chart_first_absolute_count_-_eos,chart_first_absolute_count_-_lymphs,chart_first_absolute_count_-_monos,chart_first_absolute_count_-_neuts,chart_first_alkaline_phosphate,chart_first_alt,chart_first_anion_gap,chart_first_apnea_interval,chart_first_arctic_sun/alsius_set_temp,chart_first_arctic_sun/alsius_temp_#1_c,chart_first_arctic_sun_water_temp,chart_first_arterial_base_excess,chart_first_arterial_co2_pressure,chart_first_arterial_o2_pressure,chart_first_ast,chart_first_bis_index_range,chart_first_bun,chart_first_calcium_non-ionized,chart_first_chloride_(serum),chart_first_ck-mb_fraction_(%),chart_first_ck_(cpk),chart_first_co2_production,chart_first_creatinine_(serum),chart_first_cuff_pressure,chart_first_daily_weight,chart_first_differential-basos,chart_first_differential-eos,chart_first_differential-lymphs,chart_first_differential-monos,chart_first_differential-neuts,chart_first_differential_-_immature_granulocytes,chart_first_etco2,chart_first_flow_rate_(l/min),chart_first_fspn_high,chart_first_gi_#1_tube_mark_(cm),...,med_sarna_lotion,med_senna,med_sertraline,med_sevelamer_carbonate,med_simvastatin,med_sodium_bicarbonate,med_sodium_chloride_0.9%__flush,med_sodium_chloride_0.9%__flush_for_crrt,med_sodium_chloride_3%_inhalation_soln,med_sodium_citrate_4%,med_sodium_phosphate,med_sugammadex,med_tamsulosin,med_thiamine,med_thiamine_or_placebo,med_ticagrelor,med_tiotropium_bromide,med_tirofiban,med_tobramycin,med_topiramate_(topamax),med_trazodone,med_triamcinolone_acetonide_0.1%_ointment,med_trihexyphenidyl,med_ursodiol,med_valproate_sodium,med_vancomycin,med_vancomycin_oral_liquid,med_vasopressin,med_vecuronium_bromide,med_vitamin_d,med_warfarin,other_underlying_condition,underlying_cardiac_condition,following_other_sur,following_cardiac_sur,first_mGCS_time,first_mGCS,last_mGCS_time,last_mGCS,hypothermia
0,10004720,1,61,1,0.03,0.00,0.58,1.29,13.53,149.0,13.0,14.0,20.0,35.0,34.7,35.3,0.0,42.0,64.0,32.0,10.0,14.0,8.4,99.0,13.4,307.0,140.0,0.2,26.0,70.0,0.2,0.0,3.7,8.3,87.1,0.7,30.0,47.2,30.0,58.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,5.0,1,6485.0,2,1
1,10010471,0,89,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.5,3034.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,0,0,17.0,6,6717.0,6,0
2,10038688,1,46,1,NaN,NaN,NaN,NaN,NaN,190.0,188.0,23.0,20.0,35.0,36.4,20.3,-11.0,29.0,528.0,512.0,NaN,20.0,6.6,108.0,2.1,1484.0,NaN,1.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.8,46.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,37.0,3,23026.0,1,1
3,10067389,1,76,1,NaN,NaN,NaN,NaN,NaN,307.0,1034.0,16.0,15.0,35.0,31.4,37.9,-4.0,31.0,232.0,1498.0,NaN,58.0,8.5,116.0,8.2,671.0,NaN,3.5,NaN,68.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1,0,0,0,82.0,1,18485.0,1,0
4,10079545,0,89,1,0.04,0.01,0.65,0.70,13.04,115.0,89.0,16.0,20.0,34.0,33.3,41.9,NaN,NaN,NaN,109.0,46.0,15.0,8.2,92.0,2.9,444.0,NaN,0.7,NaN,NaN,0.3,0.1,4.5,4.8,89.3,1.0,NaN,NaN,10.0,71.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,56.0,1,928.0,1,0


Identifier columns in predictors: ['subject_id']
Rhythm-like columns already in predictors: 29


,column
0,chart_cardiovascular_Regular rate and rhythm
1,chart_cv_-_past_medical_history_Arrhythmias
2,chart_ectopy_frequency_1_Runs Vtach
3,chart_ectopy_frequency_2_Runs Vtach
4,chart_heart_rhythm_1st AV (First degree AV Blo...
5,chart_heart_rhythm_2nd AV M2 (Second degree AV...
6,chart_heart_rhythm_3rd AV (Complete Heart Block)
7,chart_heart_rhythm_A Flut (Atrial Flutter)
8,chart_heart_rhythm_A Paced
9,chart_heart_rhythm_AF (Atrial Fibrillation)


,column,nonmissing,positive_or_nonempty
20,chart_heart_rhythm_SR (Sinus Rhythm),0.679677,0.414536
21,chart_heart_rhythm_ST (Sinus Tachycardia),0.679677,0.199192
19,chart_heart_rhythm_SB (Sinus Bradycardia),0.679677,0.131898
9,chart_heart_rhythm_AF (Atrial Fibrillation),0.679677,0.111709
26,chart_impaired_tissue_perfusion_ncp_-_interven...,0.679677,0.061911
23,chart_heart_rhythm_V Paced,0.679677,0.041723
1,chart_cv_-_past_medical_history_Arrhythmias,0.679677,0.034993
2,chart_ectopy_frequency_1_Runs Vtach,0.679677,0.025572
10,chart_heart_rhythm_AV Paced,0.679677,0.016151
25,chart_heart_rhythm_VT (Ventricular Tachycardia),0.679677,0.012113


In [5]:
icustays = read_table(ICU, "icustays")
needed_icu_cols = ["subject_id", "hadm_id", "stay_id", "intime", "outtime", "los"]
icustays = icustays[[c for c in needed_icu_cols if c in icustays.columns]].copy()
icustays["intime"] = pd.to_datetime(icustays["intime"], errors="coerce")
if "outtime" in icustays.columns:
    icustays["outtime"] = pd.to_datetime(icustays["outtime"], errors="coerce")

if predictors.empty:
    cohort = icustays.iloc[0:0].copy()
elif "stay_id" in predictors.columns:
    cohort = predictors[[c for c in ["subject_id", "hadm_id", "stay_id"] if c in predictors.columns]].drop_duplicates()
    cohort = cohort.merge(icustays, on=[c for c in ["subject_id", "hadm_id", "stay_id"] if c in cohort.columns and c in icustays.columns], how="left")
elif "hadm_id" in predictors.columns:
    cohort = predictors[[c for c in ["subject_id", "hadm_id"] if c in predictors.columns]].drop_duplicates()
    cohort = cohort.merge(icustays, on=[c for c in ["subject_id", "hadm_id"] if c in cohort.columns and c in icustays.columns], how="left")
else:
    cohort = predictors[["subject_id"]].drop_duplicates().merge(icustays, on="subject_id", how="left")

cohort = cohort.dropna(subset=["stay_id"]).copy() if "stay_id" in cohort.columns else cohort
if "stay_id" in cohort.columns:
    cohort["stay_id"] = cohort["stay_id"].astype(int)
if "hadm_id" in cohort.columns and cohort["hadm_id"].notna().any():
    cohort["hadm_id"] = cohort["hadm_id"].astype("Int64")
print("Cohort stays with ICU intime:", cohort.shape)
display(cohort.head())
cohort.to_csv(OUTPUT_DIR / "mimic_initial_rhythm_cohort_ids.csv", index=False)

Cohort stays with ICU intime: (1363, 6)


,subject_id,hadm_id,stay_id,intime,outtime,los
0,10004720,22081550,35009126,2186-11-12 19:55:00,2186-11-17 21:15:55,5.056192
1,10010471,29842315,32119961,2155-12-02 20:33:00,2155-12-07 18:19:18,4.907153
2,10038688,25926997,33804579,2181-12-06 08:14:00,2181-12-22 20:27:01,16.509039
3,10067389,23577021,34081592,2116-02-10 23:55:00,2116-02-24 04:49:08,13.204259
4,10079545,26201323,35535213,2155-12-25 04:32:00,2155-12-25 23:18:26,0.782245


## Search MIMIC Dictionaries for Rhythm-Like Items

The most important check is whether MIMIC has a direct item such as `Initial Rhythm`. If only generic `Heart Rhythm` telemetry items appear, that is not equivalent to initial arrest rhythm.

In [6]:
d_items = read_table(ICU, "d_items")
item_cols = [c for c in ["itemid", "label", "abbreviation", "category", "linksto", "param_type", "unitname"] if c in d_items.columns]

rhythm_item_regex = (
    r"initial.*rhythm|arrest.*rhythm|code.*rhythm|presenting.*rhythm|"
    r"heart rhythm|cardiac rhythm|\brhythm\b|"
    r"ventricular fibrillation|\bvf\b|\bv fib\b|vfib|"
    r"ventricular tachycardia|\bvt\b|\bv tach\b|vtach|torsade|"
    r"pulseless electrical|\bpea\b|asystole|asystolia|defib|defibrillat"
)

searchable = optional_text_col(d_items, "label") + " " + optional_text_col(d_items, "abbreviation") + " " + optional_text_col(d_items, "category")
dictionary_hits = d_items[searchable.str.contains(rhythm_item_regex, case=False, regex=True, na=False)][item_cols].copy()
dictionary_hits = dictionary_hits.sort_values([c for c in ["linksto", "category", "label"] if c in dictionary_hits.columns])
print("Dictionary rhythm-like hits:", len(dictionary_hits))
display(dictionary_hits)
dictionary_hits.to_csv(OUTPUT_DIR / "mimic_initial_rhythm_dictionary_hits.csv", index=False)

chart_itemids = dictionary_hits.loc[dictionary_hits.get("linksto", pd.Series(index=dictionary_hits.index)).eq("chartevents"), "itemid"].dropna().astype(int).unique().tolist()
print("Candidate chartevents itemids:", chart_itemids)

Dictionary rhythm-like hits: 4


,itemid,label,abbreviation,category,linksto,param_type,unitname
681,224421,Spont Vt,Spont Vt,Respiratory,chartevents,Numeric,mL
838,224743,Vd/Vt Ratio,Vd/Vt Ratio,Respiratory,chartevents,Numeric,NaN
5,220048,Heart Rhythm,Heart Rhythm,Routine Vital Signs,chartevents,Text,NaN
1318,225464,Cardioversion/Defibrillation,Cardioversion/Defibrillation,3-Significant Events,procedureevents,Processes,NaN


Candidate chartevents itemids: [224421, 224743, 220048]


## Stream Charted Rhythm-Like Events

This streams `chartevents` only for candidate itemids and current cohort stays. It keeps the earliest event in a configurable window around ICU admission.

In [7]:
SHOCKABLE_PATTERNS = [
    (re.compile(r"\b(v\s*[- ]?fib|vfib|ventricular fibrillation)\b", re.I), "vf"),
    (re.compile(r"\b(v\s*[- ]?tach|vtach|ventricular tachycardia|torsade)\b", re.I), "vt"),
]
NONSHOCKABLE_PATTERNS = [
    (re.compile(r"\bpea\b|pulseless electrical", re.I), "pea"),
    (re.compile(r"asystole|asystolia", re.I), "asystole"),
]


def classify_rhythm_value(value):
    text = normalize_text(value)
    if not text:
        return "missing"
    for pattern, label in NONSHOCKABLE_PATTERNS:
        if pattern.search(text):
            return label
    for pattern, label in SHOCKABLE_PATTERNS:
        if pattern.search(text):
            return label
    return "other_charted_rhythm"


def shockable_from_class(label):
    if label in {"vf", "vt"}:
        return 1
    if label in {"pea", "asystole"}:
        return 0
    return np.nan

In [8]:
chart_events_path = resolve_table(ICU, "chartevents")
chart_cols = table_columns(chart_events_path)
desired_cols = ["subject_id", "hadm_id", "stay_id", "charttime", "itemid", "value", "valuenum"]
usecols = [c for c in desired_cols if c in chart_cols]

rhythm_events = []
if not chart_itemids:
    print("No candidate chartevents itemids found.")
elif cohort.empty or "stay_id" not in cohort.columns:
    print("No cohort stay_id list available. Cannot run cohort-level chartevents extraction.")
else:
    cohort_stays = set(cohort["stay_id"].dropna().astype(int))
    itemid_set = set(int(x) for x in chart_itemids)
    for i, chunk in enumerate(pd.read_csv(chart_events_path, compression="infer", chunksize=CHUNK_SIZE, usecols=usecols), start=1):
        chunk = chunk[chunk["itemid"].isin(itemid_set)]
        if "stay_id" in chunk.columns:
            chunk = chunk[chunk["stay_id"].isin(cohort_stays)]
        if chunk.empty:
            continue
        rhythm_events.append(chunk)
        if i % 25 == 0:
            print(f"processed {i:,} chunks; retained rows so far: {sum(len(x) for x in rhythm_events):,}")

chart_rhythm = pd.concat(rhythm_events, ignore_index=True) if rhythm_events else pd.DataFrame(columns=usecols)
print("Retained chart rhythm-like rows:", len(chart_rhythm))
if not chart_rhythm.empty:
    chart_rhythm = chart_rhythm.merge(dictionary_hits[["itemid", "label", "category", "linksto"]], on="itemid", how="left")
    chart_rhythm["charttime"] = pd.to_datetime(chart_rhythm["charttime"], errors="coerce")
    chart_rhythm = chart_rhythm.merge(cohort[["stay_id", "intime"]].drop_duplicates(), on="stay_id", how="left")
    chart_rhythm["minutes_from_icu_intime"] = (chart_rhythm["charttime"] - chart_rhythm["intime"]).dt.total_seconds() / 60.0
    chart_rhythm["rhythm_class"] = chart_rhythm.get("value", pd.Series(index=chart_rhythm.index)).map(classify_rhythm_value)
    chart_rhythm["rhythm_shockable"] = chart_rhythm["rhythm_class"].map(shockable_from_class)
    in_window = chart_rhythm["minutes_from_icu_intime"].between(-60 * WINDOW_HOURS_PRE, 60 * WINDOW_HOURS_POST, inclusive="both")
    chart_rhythm_window = chart_rhythm[in_window].copy()
else:
    chart_rhythm_window = chart_rhythm.copy()

display(chart_rhythm.head(20))
display(chart_rhythm_window["rhythm_class"].value_counts(dropna=False).rename_axis("rhythm_class").reset_index(name="rows") if not chart_rhythm_window.empty else chart_rhythm_window)
chart_rhythm.to_csv(OUTPUT_DIR / "mimic_initial_rhythm_events_all_candidate_items.csv", index=False)
chart_rhythm_window.to_csv(OUTPUT_DIR / "mimic_initial_rhythm_events_in_window.csv", index=False)

processed 50 chunks; retained rows so far: 33,038
processed 75 chunks; retained rows so far: 48,061
processed 100 chunks; retained rows so far: 64,421
processed 125 chunks; retained rows so far: 79,907
processed 150 chunks; retained rows so far: 94,414
processed 175 chunks; retained rows so far: 108,187
processed 200 chunks; retained rows so far: 119,241
processed 225 chunks; retained rows so far: 130,928
processed 250 chunks; retained rows so far: 148,451
processed 275 chunks; retained rows so far: 169,934
processed 300 chunks; retained rows so far: 187,382
Retained chart rhythm-like rows: 194193


,subject_id,hadm_id,stay_id,charttime,itemid,value,valuenum,label,category,linksto,intime,minutes_from_icu_intime,rhythm_class,rhythm_shockable
0,10004720,22081550,35009126,2186-11-16 19:00:00,220048,SR (Sinus Rhythm),NaN,Heart Rhythm,Routine Vital Signs,chartevents,2186-11-12 19:55:00,5705.0,other_charted_rhythm,NaN
1,10004720,22081550,35009126,2186-11-16 20:00:00,220048,SR (Sinus Rhythm),NaN,Heart Rhythm,Routine Vital Signs,chartevents,2186-11-12 19:55:00,5765.0,other_charted_rhythm,NaN
2,10004720,22081550,35009126,2186-11-16 21:00:00,220048,SR (Sinus Rhythm),NaN,Heart Rhythm,Routine Vital Signs,chartevents,2186-11-12 19:55:00,5825.0,other_charted_rhythm,NaN
3,10004720,22081550,35009126,2186-11-16 22:00:00,220048,SR (Sinus Rhythm),NaN,Heart Rhythm,Routine Vital Signs,chartevents,2186-11-12 19:55:00,5885.0,other_charted_rhythm,NaN
4,10004720,22081550,35009126,2186-11-16 23:00:00,220048,SR (Sinus Rhythm),NaN,Heart Rhythm,Routine Vital Signs,chartevents,2186-11-12 19:55:00,5945.0,other_charted_rhythm,NaN
5,10004720,22081550,35009126,2186-11-17 00:00:00,220048,SR (Sinus Rhythm),NaN,Heart Rhythm,Routine Vital Signs,chartevents,2186-11-12 19:55:00,6005.0,other_charted_rhythm,NaN
6,10004720,22081550,35009126,2186-11-17 01:00:00,220048,SR (Sinus Rhythm),NaN,Heart Rhythm,Routine Vital Signs,chartevents,2186-11-12 19:55:00,6065.0,other_charted_rhythm,NaN
7,10004720,22081550,35009126,2186-11-17 02:00:00,220048,SR (Sinus Rhythm),NaN,Heart Rhythm,Routine Vital Signs,chartevents,2186-11-12 19:55:00,6125.0,other_charted_rhythm,NaN
8,10004720,22081550,35009126,2186-11-17 03:00:00,220048,SR (Sinus Rhythm),NaN,Heart Rhythm,Routine Vital Signs,chartevents,2186-11-12 19:55:00,6185.0,other_charted_rhythm,NaN
9,10004720,22081550,35009126,2186-11-17 04:00:00,220048,SR (Sinus Rhythm),NaN,Heart Rhythm,Routine Vital Signs,chartevents,2186-11-12 19:55:00,6245.0,other_charted_rhythm,NaN


,rhythm_class,rows
0,other_charted_rhythm,31825
1,vt,86
2,asystole,29
3,vf,12


In [9]:
if chart_rhythm_window.empty:
    first_chart_rhythm = pd.DataFrame(columns=[
        "stay_id", "initial_rhythm_class", "initial_rhythm_shockable",
        "initial_rhythm_value", "initial_rhythm_itemid", "initial_rhythm_item_label",
        "initial_rhythm_charttime", "initial_rhythm_minutes_from_icu_intime",
        "initial_rhythm_source"
    ])
else:
    priority = {
        "vf": 0,
        "vt": 0,
        "pea": 0,
        "asystole": 0,
        "other_charted_rhythm": 1,
        "missing": 2,
    }
    tmp = chart_rhythm_window.copy()
    tmp["class_priority"] = tmp["rhythm_class"].map(priority).fillna(9)
    tmp["abs_minutes_from_icu_intime"] = tmp["minutes_from_icu_intime"].abs()
    tmp = tmp.sort_values(["stay_id", "class_priority", "abs_minutes_from_icu_intime", "charttime"])
    first_chart_rhythm = tmp.groupby("stay_id", as_index=False).first()
    first_chart_rhythm = first_chart_rhythm.rename(columns={
        "rhythm_class": "initial_rhythm_class",
        "rhythm_shockable": "initial_rhythm_shockable",
        "value": "initial_rhythm_value",
        "itemid": "initial_rhythm_itemid",
        "label": "initial_rhythm_item_label",
        "charttime": "initial_rhythm_charttime",
        "minutes_from_icu_intime": "initial_rhythm_minutes_from_icu_intime",
    })
    first_chart_rhythm["initial_rhythm_source"] = "chartevents_near_icu_intime"
    keep = [
        "stay_id", "initial_rhythm_class", "initial_rhythm_shockable",
        "initial_rhythm_value", "initial_rhythm_itemid", "initial_rhythm_item_label",
        "initial_rhythm_charttime", "initial_rhythm_minutes_from_icu_intime",
        "initial_rhythm_source"
    ]
    first_chart_rhythm = first_chart_rhythm[keep]

print("Stays with charted rhythm-like event in window:", first_chart_rhythm["stay_id"].nunique() if "stay_id" in first_chart_rhythm.columns else 0)
display(first_chart_rhythm.head())
display(first_chart_rhythm["initial_rhythm_class"].value_counts(dropna=False).rename_axis("initial_rhythm_class").reset_index(name="stays") if not first_chart_rhythm.empty else first_chart_rhythm)
first_chart_rhythm.to_csv(OUTPUT_DIR / "mimic_initial_rhythm_by_stay_from_chartevents.csv", index=False)

Stays with charted rhythm-like event in window: 1347


,stay_id,initial_rhythm_class,initial_rhythm_shockable,initial_rhythm_value,initial_rhythm_itemid,initial_rhythm_item_label,initial_rhythm_charttime,initial_rhythm_minutes_from_icu_intime,initial_rhythm_source
0,30024511,other_charted_rhythm,NaN,SR (Sinus Rhythm),220048,Heart Rhythm,2176-08-20 22:15:00,64.533333,chartevents_near_icu_intime
1,30024934,other_charted_rhythm,NaN,SR (Sinus Rhythm),220048,Heart Rhythm,2143-05-01 19:04:00,93.000000,chartevents_near_icu_intime
2,30031418,other_charted_rhythm,NaN,ST (Sinus Tachycardia),220048,Heart Rhythm,2156-03-05 14:39:00,28.000000,chartevents_near_icu_intime
3,30031755,other_charted_rhythm,NaN,AF (Atrial Fibrillation),220048,Heart Rhythm,2124-04-19 18:32:00,24.766667,chartevents_near_icu_intime
4,30034142,vt,1.0,VT (Ventricular Tachycardia),220048,Heart Rhythm,2156-12-21 16:10:00,195.266667,chartevents_near_icu_intime


,initial_rhythm_class,stays
0,other_charted_rhythm,1281
1,vt,37
2,asystole,24
3,vf,5


## ICD Diagnosis Evidence for Rhythm

ICD codes can indicate diagnoses such as ventricular fibrillation/flutter or cardiac arrest, but they are admission-level and not a timestamped initial arrest rhythm. Treat these as secondary sensitivity/coverage flags, not definitive initial rhythm.

In [10]:
d_icd = read_table(HOSP, "d_icd_diagnoses", dtype={"icd_code": str})
dx = read_table(HOSP, "diagnoses_icd", dtype={"icd_code": str})
dx["icd_code_clean"] = dx["icd_code"].astype(str).str.replace(".", "", regex=False).str.upper().str.strip()
dx = dx.merge(d_icd[["icd_code", "icd_version", "long_title"]], on=["icd_code", "icd_version"], how="left")

def flag_dx(row):
    code = str(row.get("icd_code_clean", "")).upper()
    version = row.get("icd_version")
    title = normalize_text(row.get("long_title", ""))
    flags = {
        "dx_cardiac_arrest": False,
        "dx_vf": False,
        "dx_vt": False,
        "dx_pea_or_asystole": False,
    }
    if (version == 9 and code.startswith("4275")) or (version == 10 and code.startswith("I46")) or "cardiac arrest" in title:
        flags["dx_cardiac_arrest"] = True
    if (version == 9 and code.startswith("42741")) or (version == 10 and code.startswith("I4901")) or "ventricular fibrillation" in title or "ventricular flutter" in title:
        flags["dx_vf"] = True
    if (version == 9 and code.startswith("4271")) or (version == 10 and (code.startswith("I470") or code.startswith("I472"))) or "ventricular tachycardia" in title or "torsade" in title:
        flags["dx_vt"] = True
    if "asystole" in title or "pulseless electrical" in title:
        flags["dx_pea_or_asystole"] = True
    return pd.Series(flags)

dx_flags_long = pd.concat([dx[["subject_id", "hadm_id", "icd_code", "icd_version", "long_title"]], dx.apply(flag_dx, axis=1)], axis=1)
dx_hits = dx_flags_long[["dx_cardiac_arrest", "dx_vf", "dx_vt", "dx_pea_or_asystole"]].any(axis=1)
dx_hits = dx_flags_long[dx_hits].copy()
dx_by_hadm = dx_hits.groupby(["subject_id", "hadm_id"], as_index=False)[["dx_cardiac_arrest", "dx_vf", "dx_vt", "dx_pea_or_asystole"]].max()

if not cohort.empty and "hadm_id" in cohort.columns:
    cohort_dx = cohort[["subject_id", "hadm_id", "stay_id"]].drop_duplicates().merge(dx_by_hadm, on=["subject_id", "hadm_id"], how="left")
else:
    cohort_dx = dx_by_hadm.copy()

for col in ["dx_cardiac_arrest", "dx_vf", "dx_vt", "dx_pea_or_asystole"]:
    cohort_dx[col] = cohort_dx[col].fillna(False).astype(bool)
cohort_dx["dx_shockable_rhythm"] = cohort_dx["dx_vf"] | cohort_dx["dx_vt"]
cohort_dx["dx_nonshockable_rhythm"] = cohort_dx["dx_pea_or_asystole"]

print("ICD rhythm/arrest hit rows:", len(dx_hits))
display(dx_hits.head(50))
display(cohort_dx[["dx_cardiac_arrest", "dx_vf", "dx_vt", "dx_pea_or_asystole", "dx_shockable_rhythm", "dx_nonshockable_rhythm"]].mean().rename("proportion").reset_index())
dx_hits.to_csv(OUTPUT_DIR / "mimic_initial_rhythm_icd_hits_long.csv", index=False)
cohort_dx.to_csv(OUTPUT_DIR / "mimic_initial_rhythm_icd_flags_by_stay.csv", index=False)

ICD rhythm/arrest hit rows: 12185


,subject_id,hadm_id,icd_code,icd_version,long_title,dx_cardiac_arrest,dx_vf,dx_vt,dx_pea_or_asystole
462,10001401,24818636,I471,10,Supraventricular tachycardia,False,False,True,False
759,10001884,26184834,I469,10,"Cardiac arrest, cause unspecified",True,False,False,False
1997,10003019,22774359,4270,9,Paroxysmal supraventricular tachycardia,False,False,True,False
2613,10004235,22187210,V1253,9,Personal history of sudden cardiac arrest,True,False,False,False
2636,10004235,24181354,V1253,9,Personal history of sudden cardiac arrest,True,False,False,False
2764,10004401,22869003,V1253,9,Personal history of sudden cardiac arrest,True,False,False,False
2782,10004401,23920883,V1253,9,Personal history of sudden cardiac arrest,True,False,False,False
2825,10004401,25777141,V1253,9,Personal history of sudden cardiac arrest,True,False,False,False
2873,10004401,26488315,V1253,9,Personal history of sudden cardiac arrest,True,False,False,False
2893,10004401,27939719,V1253,9,Personal history of sudden cardiac arrest,True,False,False,False


,index,proportion
0,dx_cardiac_arrest,0.531181
1,dx_vf,0.140866
2,dx_vt,0.190022
3,dx_pea_or_asystole,0.000000
4,dx_shockable_rhythm,0.270726
5,dx_nonshockable_rhythm,0.000000


## Merge-Ready Feature File

The file below can be joined back to `MIMIC_Predictors.csv` by `stay_id` when available. If only `subject_id`/`hadm_id` exist in the predictors file, use those columns for the merge.

In [11]:
if cohort.empty:
    initial_rhythm_features = first_chart_rhythm.copy()
else:
    merge_cols = [c for c in ["subject_id", "hadm_id", "stay_id"] if c in cohort.columns]
    initial_rhythm_features = cohort[merge_cols].drop_duplicates().copy()
    if "stay_id" in initial_rhythm_features.columns and "stay_id" in first_chart_rhythm.columns:
        initial_rhythm_features = initial_rhythm_features.merge(first_chart_rhythm, on="stay_id", how="left")
    if "stay_id" in initial_rhythm_features.columns and "stay_id" in cohort_dx.columns:
        initial_rhythm_features = initial_rhythm_features.merge(cohort_dx.drop(columns=[c for c in ["subject_id", "hadm_id"] if c in cohort_dx.columns], errors="ignore"), on="stay_id", how="left")
    elif {"subject_id", "hadm_id"}.issubset(initial_rhythm_features.columns) and {"subject_id", "hadm_id"}.issubset(cohort_dx.columns):
        initial_rhythm_features = initial_rhythm_features.merge(cohort_dx, on=["subject_id", "hadm_id"], how="left")

for col in ["dx_cardiac_arrest", "dx_vf", "dx_vt", "dx_pea_or_asystole", "dx_shockable_rhythm", "dx_nonshockable_rhythm"]:
    if col in initial_rhythm_features.columns:
        initial_rhythm_features[col] = initial_rhythm_features[col].fillna(False).astype(int)

if "initial_rhythm_class" in initial_rhythm_features.columns:
    initial_rhythm_features["has_direct_initial_rhythm_like_chart_event"] = initial_rhythm_features["initial_rhythm_class"].notna().astype(int)
    initial_rhythm_features["has_specific_arrest_rhythm_chart_event"] = initial_rhythm_features["initial_rhythm_class"].isin(["vf", "vt", "pea", "asystole"]).astype(int)
else:
    initial_rhythm_features["has_direct_initial_rhythm_like_chart_event"] = 0
    initial_rhythm_features["has_specific_arrest_rhythm_chart_event"] = 0

initial_rhythm_features.to_csv(FEATURES_CSV, index=False)
initial_rhythm_features.to_csv(OUTPUT_DIR / "MIMIC_initial_rhythm_features.csv", index=False)
print("Wrote", FEATURES_CSV, initial_rhythm_features.shape)
display(initial_rhythm_features.head())

if not predictors.empty:
    join_cols = [c for c in ["stay_id", "hadm_id", "subject_id"] if c in predictors.columns and c in initial_rhythm_features.columns]
    merged = predictors.merge(initial_rhythm_features, on=join_cols, how="left", suffixes=("", "_initial_rhythm"))
    merged.to_csv(MERGED_PREDICTORS_CSV, index=False)
    print("Wrote", MERGED_PREDICTORS_CSV, merged.shape, "join_cols:", join_cols)

Wrote MIMIC_initial_rhythm_features.csv (1363, 19)


,subject_id,hadm_id,stay_id,initial_rhythm_class,initial_rhythm_shockable,initial_rhythm_value,initial_rhythm_itemid,initial_rhythm_item_label,initial_rhythm_charttime,initial_rhythm_minutes_from_icu_intime,initial_rhythm_source,dx_cardiac_arrest,dx_vf,dx_vt,dx_pea_or_asystole,dx_shockable_rhythm,dx_nonshockable_rhythm,has_direct_initial_rhythm_like_chart_event,has_specific_arrest_rhythm_chart_event
0,10004720,22081550,35009126,other_charted_rhythm,NaN,SR (Sinus Rhythm),220048.0,Heart Rhythm,2186-11-12 20:52:00,57.0,chartevents_near_icu_intime,1,0,0,0,0,0,1,0
1,10010471,29842315,32119961,other_charted_rhythm,NaN,1st AV (First degree AV Block),220048.0,Heart Rhythm,2155-12-02 21:00:00,27.0,chartevents_near_icu_intime,1,0,0,0,0,0,1,0
2,10038688,25926997,33804579,other_charted_rhythm,NaN,ST (Sinus Tachycardia),220048.0,Heart Rhythm,2181-12-06 08:44:00,30.0,chartevents_near_icu_intime,0,0,0,0,0,0,1,0
3,10067389,23577021,34081592,other_charted_rhythm,NaN,A Paced,220048.0,Heart Rhythm,2116-02-11 01:50:00,115.0,chartevents_near_icu_intime,1,0,0,0,0,0,1,0
4,10079545,26201323,35535213,other_charted_rhythm,NaN,SB (Sinus Bradycardia),220048.0,Heart Rhythm,2155-12-25 05:24:00,52.0,chartevents_near_icu_intime,1,0,1,0,1,0,1,0


Wrote MIMIC_Predictors_with_initial_rhythm.csv (1363, 7904) join_cols: ['subject_id']


## Summary for Manuscript/Reviewer Response

Use this section after running the notebook on the cluster. The key distinction is whether `has_specific_arrest_rhythm_chart_event` has meaningful coverage.

In [12]:
summary_rows = []
n_cohort = len(initial_rhythm_features) if "initial_rhythm_features" in globals() else 0
summary_rows.append({"metric": "cohort_stays", "value": n_cohort})

if n_cohort:
    for col in [
        "has_direct_initial_rhythm_like_chart_event",
        "has_specific_arrest_rhythm_chart_event",
        "initial_rhythm_shockable",
        "dx_cardiac_arrest",
        "dx_shockable_rhythm",
        "dx_nonshockable_rhythm",
    ]:
        if col in initial_rhythm_features.columns:
            vals = pd.to_numeric(initial_rhythm_features[col], errors="coerce")
            summary_rows.append({
                "metric": col,
                "value": float(vals.mean(skipna=True)),
                "n_nonmissing": int(vals.notna().sum()),
                "n_positive": int((vals == 1).sum()),
            })

summary = pd.DataFrame(summary_rows)
display(summary)
summary.to_csv(OUTPUT_DIR / "mimic_initial_rhythm_summary.csv", index=False)

if n_cohort and "has_specific_arrest_rhythm_chart_event" in initial_rhythm_features.columns:
    coverage = initial_rhythm_features["has_specific_arrest_rhythm_chart_event"].mean()
    if coverage < 0.2:
        print(
            "Interpretation: direct VF/VT/PEA/asystole-like charted rhythm has low coverage. "
            "This is not strong enough to use as a primary initial-rhythm covariate without a major missingness caveat."
        )
    else:
        print(
            "Interpretation: direct specific arrest rhythm coverage may be usable. "
            "Inspect values/timing manually before adding it as a prespecified sensitivity covariate."
        )

,metric,value,n_nonmissing,n_positive
0,cohort_stays,1363.000000,NaN,NaN
1,has_direct_initial_rhythm_like_chart_event,0.988261,1363.0,1347.0
2,has_specific_arrest_rhythm_chart_event,0.048423,1363.0,66.0
3,initial_rhythm_shockable,0.636364,66.0,42.0
4,dx_cardiac_arrest,0.531181,1363.0,724.0
5,dx_shockable_rhythm,0.270726,1363.0,369.0
6,dx_nonshockable_rhythm,0.000000,1363.0,0.0


Interpretation: direct VF/VT/PEA/asystole-like charted rhythm has low coverage. This is not strong enough to use as a primary initial-rhythm covariate without a major missingness caveat.
